In [87]:
import pandas as pd
import numpy as np
import re

In [88]:
import pandas as pd

columns_to_preserve = [
    'cat1', 'cat2', 'cat3', 'catsel_dob_c',
    'catsel_dob_d', 'dob_flag_c', 'dob_flag_d', 'catsel_c',
    'catsel_d', 'cut_off_c', 'cut_off_d'
]

converters = {col: str for col in columns_to_preserve}

candidate_df = pd.read_excel(
    r"C:\Users\SalauddinKhan\Desktop\STENO2024FINALRESULT\candidates_for_allocation_steno24.xlsx",
    converters=converters
)


In [89]:
#candidate_df = pd.read_csv(r"C:\Users\SalauddinKhan\Desktop\STENO2024FINALRESULT\candidates_for_allocation_steno24.csv")
vacancy_df = pd.read_csv(r"C:\Users\SalauddinKhan\Desktop\STENO2024FINALRESULT\vacancy_table_for_steno2024.csv")

In [2]:
#vacancy_df = pd.read_csv(r"C:\Users\AshutoshMishra\Downloads\vacancy_table_steno2023.csv")

In [90]:
candidate_df.columns

Index(['reg_no', 'rollno', 'cand_name', 'dob', 'gender', 'cat1', 'cat2',
       'cat3', 'exs_reservation', 'part1_gi', 'part2_ga', 'total',
       'post_apply', 'skill_medium', 'merit', 'dob_flag_c', 'dob_flag_c-2',
       'catsel_dob_c', 'catsel_dob_d', 'cut_off_c', 'cut_off_d', 'catsel_c',
       'catsel_d', 'post_preference'],
      dtype='object')

In [91]:
candidate_df['dob'] = pd.to_datetime(candidate_df['dob'])

In [92]:
cand = candidate_df.copy()
vac = vacancy_df.copy()

In [93]:
candidate_df.shape

(15364, 24)

In [94]:
cand['gender'].dtype

dtype('int64')

In [95]:
candidate_df['catsel_d'].unique()

array([nan, '9', '96', '6', '91', '90', '0', '2', '1', '92', '64', '97',
       '907', '4', '967', '94', '63', '67', '07', '7', '8', '17', '5',
       '3'], dtype=object)

In [96]:
cand['catsel_d'].value_counts()

catsel_d
9      1173
96      886
6       742
1       563
90      393
91      385
0       318
2       144
92       56
4        14
7         9
5         5
94        4
64        2
8         2
967       1
907       1
97        1
63        1
07        1
67        1
17        1
3         1
Name: count, dtype: int64

In [97]:
candidate_df['catsel_c'].unique()

array([nan, '9', '96', '91', '0', '90', '6', '94', '1', '2', '92', '64',
       '7', '4', '8', '5'], dtype=object)

In [98]:
cand['catsel_c'].value_counts()

catsel_c
9     480
96    297
6     209
1     163
90    117
91    105
0      93
2      70
4       7
92      6
7       5
5       4
94      2
64      1
8       1
Name: count, dtype: int64

In [99]:
cand[['allocated_category', 'allocated_post', 'allocated_against_ur', 'allocated_medium']] = None

In [100]:

mask = cand['post_preference'].isnull()
cand.loc[mask, 'post_preference'] = ''
cand.loc[~mask, 'post_preference'] = cand.loc[~mask, 'post_preference'].astype(str)

mask = cand['catsel_c'].isnull()
cand.loc[mask, 'catsel_c'] = ''
cand.loc[~mask, 'catsel_c'] = cand.loc[~mask, 'catsel_c'].astype(str)

mask = cand['catsel_d'].isnull()
cand.loc[mask, 'catsel_d'] = ''
cand.loc[~mask, 'catsel_d'] = cand.loc[~mask, 'catsel_d'].astype(str)


In [101]:

vac['allocated_hc'] = 0
vac['allocated_hc_prev'] = 0


In [102]:
vac['left_vacancy'] = vac['current']
vacancy_dict = {}
vac['key'] = vac['post_code'].astype(str) + vac['category_code'].astype(str) + vac['medium'].astype(str)
for index, row in vac.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()


In [ ]:
#candidate_df['ear']

In [103]:
def allocate_candidates(candidates_df, vacancy_dict):
     
    filtered_candidates = candidates_df[(candidates_df['merit'].notnull())].sort_values(by='merit')
    
    for idx, candidate in filtered_candidates.iterrows():
        allocated = False
        roll = candidate['rollno']
        post_preference = candidate['post_preference']
        DOB = candidate['dob']
        gender = candidate['gender']
        
        for post in post_preference.split(','):
            if post == 'X':
                continue
                
            medium = "X"
            
            if post.startswith("C"):
                catsel = candidate['catsel_c']
                
                if post =="C4":
                    if candidate['skill_medium'] == "English":
                        medium = "E"
                    elif candidate['skill_medium'] == "Hindi":
                        medium = "H"
                    else:
                        continue
                        
                if post in ["C2", "C3"]:
                    if candidate['skill_medium'] == "English":
                        medium = "E"
                    else:
                        continue
                        
            elif post.startswith("D"):
                catsel = candidate['catsel_d']
                
                if post == "D25" and gender == 1:
                    continue
                    
                if post in ["D14", "D41"]:
                    if candidate['skill_medium'] == "English":
                        medium = "E"
                    else:
                        continue
            
                if post == "D19":
                    if candidate['skill_medium'] == "English":
                         medium = "E"
                    elif candidate['skill_medium'] == "Hindi":
                         medium = "H"
                    else:
                        continue
                                          
            for category in catsel:
                key = str(post) + str(category) + str(medium)
                
                allocated_against_ur = ''
                
                if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                    post = vacancy_dict[key]['post_code']
                                
                    allocated = update_allocation(candidates_df, category, post, medium, allocated_against_ur, candidate, vacancy_dict)
                    if allocated:
                        vacancy_dict[key]['current'] -= 1
                        vacancy_dict[key]['allocated'] += 1
                        break
            if allocated:
                break

    return vacancy_dict




In [104]:
def update_allocation(candidates_df, category, post, medium, allocated_against_ur, candidate, vacancy_dict):
    if str(category) in ['3', '4', '5', '7', '8']:
    
        key_cat1 = str(post) + str(candidate['cat1']) + str(medium)
        
        if (key_cat1 not in vacancy_dict.keys()) or (vacancy_dict[key_cat1]['initial'] == 0):
            keyCat2= str(post) + '9' + str(medium)
            
            if (keyCat2 not in vacancy_dict.keys()) or (vacancy_dict[keyCat2]['initial'] == 0):
                return False
            else:
                if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial']:
                    vacancy_dict[keyCat2]['allocated_hc'] += 1
                allocated_against_ur = '1'
        else:
            if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial']:
                vacancy_dict[key_cat1]['allocated_hc'] += 1
            
            else :
                return False
                        
    candidates_df.loc[candidate.name, 'allocated_category'] = category
    candidates_df.loc[candidate.name, 'allocated_post'] = post
    candidates_df.loc[candidate.name, 'allocated_medium'] = medium
    candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur
    
    return True


In [105]:
def adjust_vacancy(candidates_df, vacancy_df):
    vacancy_df['current'] = vacancy_df['initial'] - vacancy_df['allocated_hc']
    vacancy_df['allocated'] = 0
    vacancy_df['left_vacancy'] = 0
        
    vacancy_df['allocated_hc_prev'] = vacancy_df['allocated_hc']
    vacancy_df['allocated_hc'] = 0

    return vacancy_df

In [111]:
 def find_lowest_marks(candidates_df, vacancy_df):
     vc = []
     vacancy_df['min_total'] = 0
     for _, row in vacancy_df.iterrows():
         lmv = {
             "post_code": str(row["post_code"]),
             "category_code": str(row["category_code"]),
             "medium": str(row["medium"]),
         }
         vc.append(lmv)
     print("Vector size--->", len(vc))
     for lmv1 in vc:
         highest_merit_candidate = get_highest_merit_candidate(candidates_df, lmv1["post_code"], lmv1["category_code"], lmv1["medium"])
         update_vacancy_table(vacancy_df, lmv1["post_code"], lmv1["category_code"], lmv1["medium"], highest_merit_candidate)
 def get_highest_merit_candidate(candidates_df, post_code, category_code, medium):
     highest_merit_candidate = None
     mask = (
         (candidates_df["allocated_post"] == str(post_code))
         & (candidates_df["allocated_category"] == str(category_code))
         & (candidates_df["allocated_medium"] == str(medium))
     )
     filtered_cand = candidates_df[mask]
     if not filtered_cand.empty:
         highest_merit_candidate = filtered_cand.loc[filtered_cand['merit'].idxmax()]
   
     return highest_merit_candidate
 def update_vacancy_table(vacancy_df, post_code, category_code, medium, highest_merit_candidate):
     key = str(post_code) + str(category_code) + str(medium)
     mask = (vacancy_df['key'] == key)
     vacancy = vacancy_df.loc[mask]
     if highest_merit_candidate is not None:
         vacancy["min_marks"] = highest_merit_candidate['total']
         vacancy["min_marks_part1_gi"] = highest_merit_candidate['part1_gi']
         vacancy["min_marks_cand_dob"] = highest_merit_candidate['dob']
         vacancy["min_marks_part2_ga"] = highest_merit_candidate['part2_ga']
         vacancy["min_marks_merit"] = highest_merit_candidate['merit']
       
         vacancy_df.loc[mask, 'min_marks'] = vacancy["min_marks"]
         vacancy_df.loc[mask, 'min_marks_part2_ga'] = vacancy["min_marks_part2_ga"]
         vacancy_df.loc[mask, 'min_marks_cand_dob'] = vacancy["min_marks_cand_dob"]
         vacancy_df.loc[mask, 'min_marks_part1_gi'] = vacancy["min_marks_part1_gi"]
         vacancy_df.loc[mask, 'min_marks_merit'] = vacancy["min_marks_merit"]

In [106]:
upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)

In [107]:
cand[cand['allocated_category'].notnull()].shape[0]

2168

In [108]:
cand[cand['allocated_category'].notnull()][['allocated_category', 'allocated_post']]

,allocated_category,allocated_post
1,9,C5
2,9,C5
4,9,D16
6,9,C9
7,9,C9
...,...,...
14841,4,D27
14925,8,D52
15237,7,D42
15311,5,C8


In [109]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')
i=0
while (updated_vacancy_df['allocated_hc_prev'] != updated_vacancy_df['allocated_hc']).any():
    i+=1
    print('Adjust Attempt-'+str(i))
    upd_vacancy_df = adjust_vacancy(cand, updated_vacancy_df)
    
    upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
    vacancy_dict = {}
    upd_vacancy_df['key'] = upd_vacancy_df['post_code'].astype(str) + upd_vacancy_df['category_code'].astype(str) + upd_vacancy_df['medium'].astype(str)
    
    for index, row in upd_vacancy_df.iterrows():
        key = row['key']
        vacancy_dict[key] = row.to_dict()
    cand[['allocated_category', 'allocated_post', 'allocated_against_ur', 'allocated_medium']] = None
    upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)
    updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

Adjust Attempt-1


In [110]:
updated_vacancy_df['left_vacancy'] = updated_vacancy_df['current']
updated_vacancy_df['current'] = updated_vacancy_df['left_vacancy'] + updated_vacancy_df['allocated']


In [ ]:
find_lowest_marks(cand, updated_vacancy_df)

In [113]:

# cand.to_excel("allocated_candidates.xl", index = False)
# cand[cand['allocated_category'].notnull()].to_csv(r"C:\Users\Aviral Chaudhary\Downloads\RP_STENO_23\only_allocated_candidates.csv", index = False)
# updated_vacancy_df.to_csv(r"C:\Users\Aviral Chaudhary\Downloads\RP_STENO_23\allocated_vacancy.csv", index = False)

cand.to_excel(r"steno2024_with_allocation.xlsx", index=False, engine='xlsxwriter')
updated_vacancy_df.to_excel(r"steno2024_with_vacancy.xlsx", index=False, engine='xlsxwriter')
